In [ ]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# =====================================================
# 2. LOAD FASHION-MNIST DATASET
# =====================================================

train = pd.read_csv("dataset/Fashion/fashion-mnist_train.csv")
test = pd.read_csv("dataset/Fashion/fashion-mnist_test.csv")

# train = pd.read_csv("dataset/Handwritten/mnist_train.csv")
# test = pd.read_csv("dataset/Handwritten/mnist_test.csv")


print("Original Training Shape:", train.shape)
print("Original Testing Shape:", test.shape)

In [ ]:
# =====================================================
# 3. DATA PREPROCESSING
# =====================================================

# Step 1: Check missing values
print("\nMissing Values in Training Data:",
      train.isnull().sum().sum())

print("Missing Values in Testing Data:",
      test.isnull().sum().sum())


# Step 2: Handle missing values if present
if train.isnull().sum().sum() > 0:
    train = train.fillna(train.median(numeric_only=True))

if test.isnull().sum().sum() > 0:
    test = test.fillna(train.median(numeric_only=True))


# Step 3: Check duplicate rows
print("\nDuplicate Rows in Training Data:",
      train.duplicated().sum())

print("Duplicate Rows in Testing Data:",
      test.duplicated().sum())


# Step 4: Remove exact duplicate rows
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)


# Step 5: Check data types
print("\nTraining Dataset Information:")
train.info()


# Step 6: Separate input features and labels
x_train = train.iloc[:, 1:].values
y_train = train.iloc[:, 0].values

x_test = test.iloc[:, 1:].values
y_test = test.iloc[:, 0].values


# Step 7: Check pixel values
print("\nMinimum Pixel Value:", x_train.min())
print("Maximum Pixel Value:", x_train.max())


# Step 8: Convert pixel values into float32
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')


# Step 9: Normalize pixel values between 0 and 1
x_train = x_train / 255.0
x_test = x_test / 255.0


# Step 10: Display final dataset dimensions
print("\nAfter Preprocessing:")
print("Training Data:", x_train.shape)
print("Training Labels:", y_train.shape)
print("Testing Data:", x_test.shape)
print("Testing Labels:", y_test.shape)



In [ ]:
# =====================================================
# 4. BUILD AND TRAIN TWO MLP MODELS
# =====================================================

histories = []
results = []

# Model configurations
configs = [(1, 'relu'), (2, 'tanh')]

for layers, activation in configs:

    # Create MLP model
    model = tf.keras.Sequential()

    # Input layer
    model.add(tf.keras.layers.Input(shape=(784,)))

    # Hidden layers
    for i in range(layers):
        model.add(tf.keras.layers.Dense(
            128, activation=activation
        ))

    # Output layer
    model.add(tf.keras.layers.Dense(
        10, activation='softmax'
    ))

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Train model
    history = model.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1
    )

    # Evaluate model
    loss, accuracy = model.evaluate(
        x_test, y_test, verbose=0
    )

    print("\nHidden Layers:", layers)
    print("Activation Function:", activation)
    print("Test Loss:", loss)
    print("Test Accuracy:", accuracy * 100, "%")

    # Store training history and accuracy
    histories.append(history)
    results.append(accuracy)


In [ ]:
# =====================================================
# 5. PLOT ACCURACY CURVES
# =====================================================

plt.figure(figsize=(8, 5))

for i, h in enumerate(histories):

    plt.plot(
        h.history['accuracy'],
        label=f'Model {i+1} Training'
    )

    plt.plot(
        h.history['val_accuracy'],
        label=f'Model {i+1} Validation'
    )

plt.title("MLP Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# =====================================================
# 6. PLOT LOSS CURVES
# =====================================================

plt.figure(figsize=(8, 5))

for i, h in enumerate(histories):

    plt.plot(
        h.history['loss'],
        label=f'Model {i+1} Training'
    )

    plt.plot(
        h.history['val_loss'],
        label=f'Model {i+1} Validation'
    )

plt.title("MLP Loss Comparison")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# =====================================================
# 7. COMPARE FINAL MODEL ACCURACY
# =====================================================

print("\nFinal Model Comparison")

print("Model 1 Accuracy:", results[0] * 100, "%")
print("Model 2 Accuracy:", results[1] * 100, "%")

if results[0] > results[1]:
    print("Best Model: Model 1")
elif results[1] > results[0]:
    print("Best Model: Model 2")
else:
    print("Both Models Have Equal Accuracy")